# 03.4 — Fixtures and parametrized tests

[Previous: 03.3](03.3-testing-behavior-with-pytest.ipynb) · [Class index and setup](README.md) · [Next: 03.5](03.5-review-and-conflict-practice.ipynb)

**Purpose:** Reuse setup while keeping each test independent and controlling external inputs.

**In class:** 0 minutes. **Independent work:** 60 minutes, within the index's 210-minute budget. The separate course project is additional workload.

## Learning outcomes

- Use a fresh fixture instead of shared mutable test state.
- Choose representative normal, empty, zero, and invalid cases.
- Replace an input function locally without bypassing schema validation.

## Starting point

You know basic Python. The complete Report/Observation package, data, practice inputs, and references are supplied in this class folder. No previous notebook, earlier exercise answer, or model/API access is needed. In VS Code, select this class's `.venv`, then use **Restart Kernel → Run All**. Every part imports its own names.

Default cells read local examples and may run local checks; they do not mutate Git repositories, change configuration, install packages, read credentials, or contact GitHub. All Git writing commands below are manual terminal work in a NEW scratch copy outside the course checkout. Empty answer cells are safe. Attempt before uncommenting `%load` in its separate cell: run to reveal source, then run that populated cell separately. Loading may replace its invoking cell; a kernel restart does not remove loaded source.

In [ ]:
from pathlib import Path
import sys

if not Path("solutions/exercise_03_4_h1.py").is_file() or not Path("report_app/models.py").is_file():
    raise FileNotFoundError(
        "Open 03-git-collaboration-and-testing.code-workspace and restart the kernel. "
        "Download the complete folder, including report_app/, practice/, and solutions/."
    )
print("Notebook kernel:", sys.executable)
print("Class assets available.")

## Independent work · first 30 minutes — Setup, cases, and boundaries

A **fixture** is named setup that pytest supplies to requesting tests. With the default function scope, each requesting test gets a fresh fixture evaluation. This is a useful default for mutable objects: one test should not depend on a previous test having filled a Report or left a list unchanged. Reusing a global mutable object can make tests pass or fail depending on order.

The supplied suite defines the following pattern. pytest reads the test's parameter name and supplies the fixture value; a normal Python call will not automatically inject it.

```python
@pytest.fixture
def observations():
    return [Observation(label="north", value=2.5),
            Observation(label="south", value=3.5)]

def test_total_uses_the_records(observations):
    assert summarize(observations)["total"] == 6
```

A fixture is not needed for every constant. Use it when setup has a clear meaning or is shared among tests. Keep it small enough that a reader can understand the starting state.

### Parametrization names the cases

`@pytest.mark.parametrize("values, expected", [([], None), ([2, 4], 3), ([0], 0)])` runs one behavior check for each input/expectation pair. These cases have different meanings: no measurements, multiple ordinary measurements, and one measured zero. Three intentionally chosen cases are more informative than many arbitrary values with no stated contract.

For invalid quantities, use `with pytest.raises(ValidationError):` around the operation expected to reject a negative value or a nonnumeric string. The schema in [models.py](report_app/models.py) defines nonnegative finite numbers and a nonempty label. Test that input rejection happens, rather than catching every exception and calling it success. For decimal arithmetic, `pytest.approx(0.3)` avoids confusing floating-point representation with a calculation defect.

In [ ]:
import pytest
from pydantic import ValidationError
from report_app.models import Observation
from report_app.analysis import summarize

for values, expected in [([], None), ([2, 4], 3), ([0], 0)]:
    rows = [Observation(label="case", value=value) for value in values]
    assert summarize(rows)["mean"] == expected
with pytest.raises(ValidationError):
    Observation(label="bad", value=-1)
rows = [Observation(label="a", value=0.1), Observation(label="b", value=0.2)]
assert summarize(rows)["total"] == pytest.approx(0.3)
print("Normal, empty, zero, invalid, and decimal cases checked.")

### Replace the input boundary, keep the real logic

[source.py](report_app/source.py) separates `read_payload()` from `load_observations()`. The first reads bundled JSON. The second calls the first and validates each record as an Observation. This is a small boundary you can replace in a test without a live service, credentials, a model, or a large mocking framework.

A pytest test can request `monkeypatch` and call `monkeypatch.setattr(source, "read_payload", fake_read)`. Patch the name where the code under test looks it up. Here `load_observations` looks up `read_payload` in the `source` module, so that is the name to replace. pytest restores the original attribute after the test.

The next ordinary notebook cell uses a context manager to demonstrate that same restoration explicitly. It supplies raw dictionaries but still executes the real validation and summary code. A fake returning already-validated results from the whole calculation would miss the boundary behavior we want to check.

In [ ]:
from report_app import source

original_reader = source.read_payload
with pytest.MonkeyPatch.context() as patch:
    patch.setattr(source, "read_payload", lambda: [{"label": "local", "value": 7}])
    loaded = source.load_observations()
    assert summarize(loaded)["mean"] == 7
assert source.read_payload is original_reader
with pytest.MonkeyPatch.context() as patch:
    patch.setattr(source, "read_payload", lambda: [{"label": "invalid", "value": -1}])
    with pytest.raises(ValidationError):
        source.load_observations()
print("Input replaced locally; validation stayed real; original reader restored.")

Open [test_source.py](tests/test_source.py). Compare its automatically provided fixture with the context above. Neither contacts a network. Later API and AI classes can preserve this habit: unit checks exercise parsing/validation with controlled inputs; a separate integration check exercises a real service when configured. A fake passing test does not establish the availability or behavior of the real provider.

### 🧩 Exercise 03.4.H1 — Write independent, representative tests

> **Independent work · 30 minutes**
>
> **Goal:** Cover the contract with fresh setup and a controlled input boundary.
>
> **Start here:** A NEW scratch copy of the complete supplied app and environment metadata. Earlier exercise answers are unnecessary. Read tests/test_reports.py and tests/test_source.py after attempting your own checks.
>
> **Task:** Write a fixture returning fresh Observations, parameter cases for empty/multiple/zero means, and an invalid-value raises check. Add one monkeypatch test returning a local raw payload and another invalid raw payload that validation rejects. Run the selected tests and the complete passing suite; explain what the replacement does and does not establish.
>
> **Check:** The passing suite stays green. Each case has a reason. Tests do not rely on order, shared mutable state, live APIs, or credentials; schema validation still executes.
>
> **Hint:** Patch source.read_payload, because source.load_observations looks up that name. Request the monkeypatch fixture in pytest; do not call a decorated fixture directly.

In [ ]:
# Your attempt for 03.4.H1.
# Record code, commands, observations, and explanation here.

In [ ]:
# Reveal only after attempting the exercise.
# Remove the leading '# ' from the next line and run this cell.
# %load solutions/exercise_03_4_h1.py

## Completion — included in the 30-minute task

Explain one normal case, one boundary case, and one rejected input. Describe why the fixture is fresh for each requesting test. Point to the exact input boundary and explain why a passing fake-input test cannot promise that a future HTTP endpoint is reachable.

**Optional extension · 20 minutes, excluded:** compare a deliberately shared mutable fixture with the default function-scoped setup in a disposable copy. Keep the final tests independent.

## References

- [pytest fixtures](https://docs.pytest.org/en/stable/how-to/fixtures.html)
- [pytest parametrization](https://docs.pytest.org/en/stable/how-to/parametrize.html)
- [pytest monkeypatch](https://docs.pytest.org/en/stable/how-to/monkeypatch.html)
- [Previous: behavior tests](03.3-testing-behavior-with-pytest.ipynb)
- [Next: review and conflict practice](03.5-review-and-conflict-practice.ipynb)
- [Class index](README.md)